# 01 — EDA and Data Quality Profiling

**Project:** dq-ml-impact-lab  
**Author:** Salini Anbalagan  
**Dataset:** UCI Bank Marketing Dataset  

---

## Notebook Objectives

1. Load and inspect the UCI Bank Marketing dataset
2. Perform exploratory data analysis (EDA) — shape, types, distributions
3. Establish **baseline data quality metrics** manually across key DQ dimensions
4. Generate an automated profiling report using `ydata-profiling`
5. Document findings that inform the degradation experiment design

---

## Banking Context

This dataset captures phone-based direct marketing campaigns by a Portuguese banking institution.  
The target variable `y` indicates whether the client subscribed to a term deposit (`yes`/`no`).  

In a real digital banking context, this mirrors **customer contact data** — a domain where  
data quality directly impacts CRM effectiveness, campaign ROI, and regulatory compliance.

> **Key question for this notebook:**  
> *What is the baseline data quality of this dataset before any degradation is applied?*

## 0. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Project utilities
import sys
sys.path.append('..')  # Allow imports from src/
from src.utils import load_dataset, validate_dataframe, print_scorecard

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 13

print('Setup complete.')

## 1. Load Dataset

In [ ]:
# Option A: Load from disk (after manual download)
DATA_PATH = '../data/raw/bank-additional-full.csv'

# Option B: Download via ucimlrepo (uncomment if not downloaded)
# from ucimlrepo import fetch_ucirepo
# repo = fetch_ucirepo(id=222)
# df = pd.concat([repo.data.features, repo.data.targets], axis=1)
# df.to_csv(DATA_PATH, sep=';', index=False)

df = load_dataset(DATA_PATH, sep=';', target_col='y')
print(f'\nDataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')

In [ ]:
# Quick validation gate
is_valid, issues = validate_dataframe(df, min_rows=100, required_cols=['y'])
if not is_valid:
    raise ValueError(f'Dataset validation failed: {issues}')
print('Validation passed.')

In [ ]:
df.head()

## 2. Basic Structure Inspection

In [ ]:
print('=== SHAPE ===')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')

print('\n=== DTYPES ===')
print(df.dtypes.value_counts())

print('\n=== COLUMN LIST ===')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2}. {col} ({df[col].dtype})')

In [ ]:
# Summary statistics — numeric columns
df.describe().T.style.background_gradient(cmap='Blues', axis=1)

In [ ]:
# Summary statistics — categorical columns
df.describe(include=['object']).T

## 3. Target Variable Analysis

Understanding class balance is critical before any ML experiment.  
Class imbalance is itself a form of data quality concern for supervised learning.

In [ ]:
target_counts = df['y'].value_counts()
target_pct = df['y'].value_counts(normalize=True).mul(100).round(2)

print('Target Distribution:')
print(pd.DataFrame({'count': target_counts, 'pct': target_pct}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
target_counts.plot(kind='bar', ax=axes[0], color=['#3498db', '#e74c3c'], edgecolor='black')
axes[0].set_title('Target Variable: Absolute Count')
axes[0].set_xlabel('Subscribed to Term Deposit')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Pie chart
axes[1].pie(
    target_counts,
    labels=target_counts.index,
    autopct='%1.1f%%',
    colors=['#3498db', '#e74c3c'],
    startangle=90
)
axes[1].set_title('Target Variable: Proportion')

plt.suptitle('Class Imbalance — UCI Bank Marketing Dataset', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../data/degraded/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nObservation: Significant class imbalance present. This is a known DQ concern for ML.')

## 4. Missing Value Analysis

The UCI Bank Marketing dataset uses the string `'unknown'` as a coded missing value  
rather than `NaN`. This is a real-world DQ pattern — **missingness masked as a category**.

In [ ]:
# Standard NaN check
null_counts = df.isnull().sum()
print('Standard NaN counts:')
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else 'No NaN values found.')

# Coded missing: 'unknown' string
print('\nCoded missing values ("unknown" string):')
unknown_counts = (df == 'unknown').sum()
unknown_nonzero = unknown_counts[unknown_counts > 0]
if unknown_nonzero.empty:
    print('No "unknown" values found.')
else:
    unknown_pct = (unknown_nonzero / len(df) * 100).round(2)
    print(pd.DataFrame({'count': unknown_nonzero, 'pct (%)': unknown_pct}))

In [ ]:
# Visualise coded missingness heatmap
unknown_mask = (df == 'unknown')
cols_with_unknown = unknown_mask.columns[unknown_mask.any()].tolist()

if cols_with_unknown:
    plt.figure(figsize=(12, 4))
    sns.heatmap(
        unknown_mask[cols_with_unknown].T,
        cbar=False,
        cmap='Reds',
        yticklabels=True
    )
    plt.title('Coded Missing Values ("unknown") by Column')
    plt.xlabel('Row Index')
    plt.tight_layout()
    plt.show()
else:
    print('No coded missing values to visualise.')

## 5. Numeric Feature Distributions

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f'Numeric columns: {numeric_cols}')

fig, axes = plt.subplots(2, len(numeric_cols), figsize=(4 * len(numeric_cols), 8))

for i, col in enumerate(numeric_cols):
    # Histogram
    axes[0, i].hist(df[col].dropna(), bins=40, color='#3498db', edgecolor='white', alpha=0.85)
    axes[0, i].set_title(f'{col} — Distribution')
    axes[0, i].set_ylabel('Frequency')

    # Box plot
    axes[1, i].boxplot(df[col].dropna(), patch_artist=True,
                       boxprops=dict(facecolor='#aed6f1', color='#2980b9'))
    axes[1, i].set_title(f'{col} — Boxplot')

plt.suptitle('Numeric Feature Distributions — Baseline', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../data/degraded/numeric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Categorical Feature Distributions

In [ ]:
cat_cols = df.select_dtypes(include=['object']).columns.drop('y').tolist()
print(f'Categorical columns (excl. target): {cat_cols}')

n_cols = 3
n_rows = (len(cat_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    vc = df[col].value_counts()
    axes[i].barh(vc.index, vc.values, color='#85c1e9', edgecolor='white')
    axes[i].set_title(f'{col}')
    axes[i].set_xlabel('Count')
    axes[i].invert_yaxis()

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Categorical Feature Distributions — Baseline', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../data/degraded/categorical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Baseline DQ Metrics (Manual)

Before running the ML profiler, we establish a manual baseline across  
four key DQ dimensions. This acts as a ground-truth reference point.

In [ ]:
baseline_dq = []

for col in df.columns:
    series = df[col]
    n = len(series)

    # Completeness: non-null rate (NaN only; 'unknown' treated separately)
    completeness = 1 - series.isnull().mean()

    # Coded missingness: 'unknown' rate
    unknown_rate = (series == 'unknown').sum() / n if series.dtype == object else 0.0

    # Uniqueness: proportion of unique values
    uniqueness = series.nunique() / n

    # Cardinality (number of unique values)
    cardinality = series.nunique()

    baseline_dq.append({
        'column': col,
        'dtype': str(series.dtype),
        'completeness': round(completeness, 4),
        'coded_missing_rate': round(unknown_rate, 4),
        'uniqueness_ratio': round(uniqueness, 4),
        'cardinality': cardinality,
        'n_nulls': int(series.isnull().sum()),
    })

baseline_dq_df = pd.DataFrame(baseline_dq)
baseline_dq_df.style.background_gradient(
    subset=['completeness'], cmap='RdYlGn'
).background_gradient(
    subset=['coded_missing_rate'], cmap='Reds'
)

In [ ]:
# Save baseline DQ metrics
Path('../data/degraded').mkdir(parents=True, exist_ok=True)
baseline_dq_df.to_csv('../data/degraded/baseline_dq_metrics.csv', index=False)
print('Baseline DQ metrics saved to data/degraded/baseline_dq_metrics.csv')

## 8. Correlation Analysis (Numeric Features)

In [ ]:
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5,
    vmin=-1, vmax=1
)
plt.title('Correlation Matrix — Numeric Features (Baseline)', fontsize=13)
plt.tight_layout()
plt.savefig('../data/degraded/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Automated Profiling Report (ydata-profiling)

Generates a comprehensive HTML report for deep-dive inspection.  
This is saved to disk — open in a browser for full interactivity.

> **Note:** This cell may take 1–3 minutes on the full dataset (45k rows).  
> Set `minimal=True` for a faster report during development.

In [ ]:
from ydata_profiling import ProfileReport

profile = ProfileReport(
    df,
    title='UCI Bank Marketing — Baseline DQ Profile',
    explorative=True,
    minimal=False,  # Set True for faster run
)

report_path = '../data/degraded/baseline_profile_report.html'
profile.to_file(report_path)
print(f'Profiling report saved to: {report_path}')
print('Open in a browser to explore interactively.')

## 10. Summary of Findings

Captures key observations that directly inform the degradation experiment design in Notebook 03.

In [ ]:
print('=' * 60)
print('  BASELINE EDA — KEY FINDINGS SUMMARY')
print('=' * 60)

print(f"""
Dataset Shape       : {df.shape[0]:,} rows x {df.shape[1]} columns
Target Balance      : {target_pct.get('yes', 0):.1f}% yes / {target_pct.get('no', 0):.1f}% no
NaN Missingness     : {df.isnull().sum().sum()} total nulls
Coded Missingness   : {(df == 'unknown').sum().sum()} 'unknown' values
Numeric Columns     : {len(numeric_cols)}
Categorical Columns : {len(cat_cols) + 1} (incl. target)
""")

print('DQ Observations:')
print('  1. Class imbalance (~88% no) — requires balanced classifiers')
print('  2. Coded missingness ("unknown") in job, education, contact, etc.')
print('  3. Numeric features show skew (duration, campaign, pdays)')
print('  4. pdays has extreme value (999 = not contacted) — domain-coded outlier')
print()
print('Degradation Experiment Implications:')
print('  - Null injection will compound existing coded missingness')
print('  - Label noise at 10%+ will significantly worsen imbalance effects')
print('  - Outlier injection on duration/campaign will be most impactful')
print('=' * 60)

---

## Next Steps

→ **Notebook 02:** `02_dq_profiler.ipynb` — Run ML-powered DQ scoring using `DQProfiler`  
→ **Notebook 03:** `03_degradation_experiments.ipynb` — Inject controlled DQ degradation  
→ **Notebook 04:** `04_ml_impact_analysis.ipynb` — Measure ML performance drop

---
*dq-ml-impact-lab | Salini Anbalagan*